Simulation d'un réseau de neurones 3x3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ThreeByThreeNN(nn.Module):
    def __init__(self):
        super(ThreeByThreeNN, self).__init__()
        self.fc1 = nn.Linear(3, 3)  # couche 1 : 3 → 3
        self.fc2 = nn.Linear(3, 3)  # couche 2 : 3 → 3
        self.fc3 = nn.Linear(3, 3)  # couche 3 : 3 → 3

    def forward(self, x):
        x = F.relu(self.fc1(x))  # activation ReLU après couche 1
        x = F.relu(self.fc2(x))  # activation ReLU après couche 2
        x = self.fc3(x)          # pas d'activation finale (à adapter selon la tâche)
        return x

Masque pour le dropout

In [11]:
p1 = 0.05 #Note : c'est la proba de dropout donc proba de masquer le neurone zij
p2 = 0.005
p3 = 0.05

class ThreeByThreeNN_MCdropout(nn.Module):
    def __init__(self, model, p1=0.05, p2=0.005, p3=0.05):
        super().__init__()
        self.model = model
        self.p1 = p1
        self.p2 = p2
        self.p3 = p3
    
    def dropout_mask(self, x, p):
        mask = (torch.rand_like(x) > p).float() / (1 - p) ##garde x1 si proba >p1, on normalise pour que l'espérance reste constante
        return mask

    def forward(self, x):
        # Couche 1 + dropout manuel
        x1 = F.relu(self.model.fc1(x))
        mask1 = self.dropout_mask(x1, self.p1)
        x1_d = x1 * mask1 #vecteur x1 auquel on a appliqué le masque, correspond à Wi

        # Couche 2 + dropout manuel
        x2 = F.relu(self.model.fc2(x1_d))
        mask2 = self.dropout_mask(x2, self.p2)
        x2_d = x2 * mask2

        # Couche 3 + dropout manuel
        x3 = self.model.fc3(x2_d)
        mask3 = self.dropout_mask(x3, self.p3)
        x3_d = x3 * mask3

        # Affichage des masques
        #print("Masque couche 1:\n", mask1)
        #print("Masque couche 2:\n", mask2)
        #print("Masque couche 3:\n", mask3)

        return x3_d


Données

In [12]:
batch_size = 50
X = torch.randn(batch_size, 3)  # batch de 50 vecteurs 3D

# Calcul des normes L2 par vecteur (dim=1)
norms = torch.norm(X, p=2, dim=1)  # shape [batch_size]

max_norm = norms.max()

# Normalisation par la norme max pour que la norme max soit 1
X = X / max_norm

print("Normes après normalisation (max):", torch.norm(X, p=2, dim=1).max().item())  # doit être 1.0

Normes après normalisation (max): 1.0


In [13]:
model_orig = ThreeByThreeNN()
model = ThreeByThreeNN_MCdropout(model_orig, p1=0.05, p2=0.005, p3=0.05)

In [14]:
model.train()

Y = model_orig(X)  # sortie batch_size x 3
Y_hat = model(X)

print("Shape X:", X.shape)
print("Shape Y:", Y.shape)
print("Shape Y_hat:", Y_hat.shape)

Shape X: torch.Size([50, 3])
Shape Y: torch.Size([50, 3])
Shape Y_hat: torch.Size([50, 3])


Calcul des métriques

In [15]:
# --- Calcul des métriques sur batch ---

def compute_ece(probs, labels, n_bins=10):
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)
    ece = torch.zeros(1)

    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            acc_in_bin = accuracies[in_bin].float().mean()
            avg_conf = confidences[in_bin].mean()
            ece += torch.abs(avg_conf - acc_in_bin) * prop_in_bin
    return ece

In [17]:
Y_class = torch.argmax(Y, dim=1)
log_probs = F.log_softmax(Y_hat, dim=1)
nll = F.nll_loss(log_probs, Y_class)

mean_probs = torch.softmax(Y_hat, dim=1)
Y_one_hot = F.one_hot(Y_class, num_classes=3).float()
brier = torch.mean(torch.sum((mean_probs - Y_one_hot) ** 2, dim=1))
ece = compute_ece(mean_probs, Y_class)

print("Negative Log-likelihood:", nll.item())
print("Brier score:", brier.item())
print("Expected Calibration Error:", ece.item())

Negative Log-likelihood: 0.9526417255401611
Brier score: 0.5663816928863525
Expected Calibration Error: 0.594153642654419


Grid search pour la probabilité de dropout optimale

In [20]:
def run_dropout_trial(p1, p2, p3, X, Y, T=1000):
    model_orig = ThreeByThreeNN()
    model = ThreeByThreeNN_MCdropout(model_orig, p1=p1, p2=p2, p3=p3)
    model.train()

    outputs = []
    with torch.no_grad():
        for _ in range(T):
            out = model(X)
            outputs.append(out.unsqueeze(0))
    outputs = torch.cat(outputs, dim=0)
    mean_pred = outputs.mean(dim=0)

    Y_class = torch.argmax(Y, dim=1)
    log_probs = F.log_softmax(mean_pred, dim=1)
    nll = F.nll_loss(log_probs, Y_class)

    mean_probs = torch.softmax(mean_pred, dim=1)
    Y_one_hot = F.one_hot(Y_class, num_classes=3).float()
    brier = torch.mean(torch.sum((mean_probs - Y_one_hot) ** 2, dim=1))
    ece = compute_ece(mean_probs, Y_class)

    return nll.item(), brier.item(), ece.item(), (p1, p2, p3)


In [24]:
import random

n_trials = 50
best_score = float('inf')
best_parameters = None

all_nll, all_brier, all_ece = [], [], []
all_results = []

for i in range(n_trials):
    p1 = random.uniform(0.1, 0.3)
    p2 = random.uniform(0.1, 0.3)
    p3 = random.uniform(0.1, 0.3)

    nll, brier, ece, params = run_dropout_trial(p1, p2, p3, X, Y, T=1000)

    all_results.append((nll, brier, ece, params))
    all_nll.append(nll)
    all_brier.append(brier)
    all_ece.append(ece)

nll_max = max(all_nll)
brier_max = max(all_brier)
ece_max = max(all_ece)

for nll, brier, ece, parameters in all_results:
    nll_norm = nll / nll_max if nll_max > 0 else 0
    brier_norm = brier / brier_max if brier_max > 0 else 0
    ece_norm = ece / ece_max if ece_max > 0 else 0

    score = (nll_norm + brier_norm + ece_norm) / 3

    if score < best_score:
        best_score = score
        best_parameters = parameters

print(f"\n Best params found: p1={best_parameters[0]:.3f}, p2={best_parameters[1]:.3f}, p3={best_parameters[2]:.3f}")
print(f"→ Best (normalized) average score = {best_score:.4f}")


 Best params found: p1=0.176, p2=0.152, p3=0.136
→ Best (normalized) average score = 0.4412


Modèle finale avec les meilleures probas

In [25]:
# Utilisation des meilleures probabilités
model_final = ThreeByThreeNN_MCdropout(ThreeByThreeNN(), *best_params)
model_final.train()  # important : on garde dropout actif

T = 1000  # nombre de passes MC Dropout
outputs = []
with torch.no_grad():
    for _ in range(T):
        outputs.append(model_final(X).unsqueeze(0))  # X peut être un seul vecteur

outputs = torch.cat(outputs, dim=0)       # shape [T, 1, 3]
mean_pred = outputs.mean(dim=0)           # moyenne des prédictions
uncertainty = outputs.var(dim=0)          # variance = incertitude

print("\nMC Dropout output (mean logits):", mean_pred)
print("MC Dropout Uncertainty:", uncertainty)  


MC Dropout output (mean logits): tensor([[-0.3046, -0.0293, -0.0398],
        [-0.3032, -0.0289, -0.0405],
        [-0.3098, -0.0289, -0.0403],
        [-0.2935, -0.0298, -0.0391],
        [-0.2793, -0.0604, -0.0617],
        [-0.3083, -0.0305, -0.0396],
        [-0.3151, -0.0299, -0.0377],
        [-0.3078, -0.0320, -0.0319],
        [-0.2987, -0.0297, -0.0395],
        [-0.3072, -0.0294, -0.0358],
        [-0.3283, -0.0413, -0.0101],
        [-0.3072, -0.0305, -0.0374],
        [-0.2965, -0.0372, -0.0456],
        [-0.3107, -0.0296, -0.0402],
        [-0.3096, -0.0286, -0.0410],
        [-0.3061, -0.0288, -0.0405],
        [-0.3056, -0.0305, -0.0403],
        [-0.2982, -0.0363, -0.0447],
        [-0.3104, -0.0296, -0.0380],
        [-0.3046, -0.0288, -0.0392],
        [-0.3067, -0.0308, -0.0393],
        [-0.3053, -0.0292, -0.0389],
        [-0.3073, -0.0301, -0.0377],
        [-0.3128, -0.0307, -0.0373],
        [-0.3138, -0.0321, -0.0325],
        [-0.3018, -0.0295, -0.0405],
    